# Фінальний проєкт: Бінарна класифікація клієнтів (Kaggle)
**Мета:** Розробка ефективної прогнозної моделі для обробки великої кількості вхідних ознак із застосуванням технік балансування класів.

## 1. Завантаження даних та імпорт бібліотек

In [1]:
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

train_df = pd.read_csv('final_proj_data.csv')
test_df = pd.read_csv('final_proj_test.csv')

X = train_df.drop(columns=['y'])
y = train_df['y']

## 2. Аналіз даних та відбір ознак (Feature Selection)
Під час первинного EDA було виявлено, що датасет є сильно розрідженим (понад 1.6 млн пропусків) та має значний дисбаланс класів (87% класу 0 проти 13% класу 1). 

**Стратегія очищення:**
1. Видалення ознак, де кількість пропусків (NaN) перевищує 80%, оскільки їх заповнення призведе до спотворення сигналу.
2. Видалення текстових категоріальних ознак із кардинальністю > 3000 (унікальні ідентифікатори/хеші), щоб уникнути перенавчання моделі.

In [2]:
# А. Аналіз та видалення колонок із критичною кількістю пропусків
missing_percent = X.isna().mean()
cols_to_drop_na = missing_percent[missing_percent > 0.80].index.tolist()

# Б. Аналіз та видалення аномально великих категорій
cat_cols_initial = X.select_dtypes(include=['object', 'string']).columns
cardinality = X[cat_cols_initial].nunique()
cols_to_drop_cardinality = cardinality[cardinality > 3000].index.tolist()

# В. Очищення датасетів
all_cols_to_drop = list(set(cols_to_drop_na + cols_to_drop_cardinality))
X = X.drop(columns=all_cols_to_drop)
test_df_clean = test_df.drop(columns=all_cols_to_drop)

print(f"Видалено ознак: {len(all_cols_to_drop)}")
print(f"Залишилось ознак для моделювання: {X.shape[1]}")

Видалено ознак: 158
Залишилось ознак для моделювання: 72


## 3. Побудова ML-конвеєра та моделювання
Для обробки залишкових даних побудовано `Pipeline`, який включає:
* **Числові ознаки:** заповнення пропусків медіаною + стандартизація.
* **Категоріальні ознаки:** кодування через `TargetEncoder` (ефективно для високої кардинальності).
* **Балансування класів:** генерація синтетичних прикладів міноритарного класу за допомогою `SMOTE`.
* **Модель:** `HistGradientBoostingClassifier`, що оптимально працює з великими масивами даних.

In [4]:
# Динамічне визначення типів колонок після очищення
cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'string']).columns.tolist()

num_transformer = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', TargetEncoder(target_type='binary', smooth='auto'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# Збірка фінального конвеєра
pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', HistGradientBoostingClassifier(random_state=42))
])

# Крос-валідація (StratifiedKFold для збереження пропорцій класів)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring='balanced_accuracy', n_jobs=-1)

print(f"Local Balanced Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Навчання на всіх даних та генерація прогнозу
pipeline.fit(X, y)
predictions = pipeline.predict(test_df_clean)

# Збереження результату
submission = pd.DataFrame({
    'index': test_df['index'] if 'index' in test_df.columns else test_df.index,
    'y': predictions
})
submission.to_csv('final_submission.csv', index=False)

Local Balanced Accuracy: 0.8893 (+/- 0.0048)
